In [1]:
import pandas as pd

df = pd.read_csv("D:/Git/esg-disclosure-gap-analyser/data/processed/SP 500 ESG Risk Ratings.csv")

print(df.shape)
print(df.columns.tolist())
print(df.head())

(503, 15)
['Symbol', 'Name', 'Address', 'Sector', 'Industry', 'Full Time Employees', 'Description', 'Total ESG Risk score', 'Environment Risk Score', 'Governance Risk Score', 'Social Risk Score', 'Controversy Level', 'Controversy Score', 'ESG Risk Percentile', 'ESG Risk Level']
  Symbol                      Name  \
0   ENPH      Enphase Energy, Inc.   
1    EMN  Eastman Chemical Company   
2    DPZ       Domino's Pizza Inc.   
3    DAY            Dayforce, Inc.   
4    DVA               Davita Inc.   

                                             Address             Sector  \
0  47281 Bayside Parkway\nFremont, CA 94538\nUnit...         Technology   
1  200 South Wilcox Drive\nKingsport, TN 37662\nU...    Basic Materials   
2  30 Frank Lloyd Wright Drive\nAnn Arbor, MI 481...  Consumer Cyclical   
3  3311 East Old Shakopee Road\nMinneapolis, MN 5...         Technology   
4  2000 16th Street\nDenver, CO 80202\nUnited States         Healthcare   

                  Industry Full Time Empl

In [2]:
print(df['Sector'].unique())
print()
print(df['Sector'].value_counts())

['Technology' 'Basic Materials' 'Consumer Cyclical' 'Healthcare'
 'Industrials' 'Utilities' 'Financial Services' 'Energy' 'Real Estate'
 'Communication Services' 'Consumer Defensive' nan]

Technology                76
Industrials               73
Financial Services        66
Healthcare                65
Consumer Cyclical         57
Consumer Defensive        36
Utilities                 32
Real Estate               31
Basic Materials           22
Energy                    22
Communication Services    22
Name: Sector, dtype: int64


In [3]:
print(df.isnull().sum())
print()
print(f"Total rows: {len(df)}")
print(f"Rows with ESG scores: {df['Total ESG Risk score'].notna().sum()}")
print(f"Rows missing ESG scores: {df['Total ESG Risk score'].isna().sum()}")

Symbol                      0
Name                        0
Address                     1
Sector                      1
Industry                    1
Full Time Employees         5
Description                 1
Total ESG Risk score       73
Environment Risk Score     73
Governance Risk Score      73
Social Risk Score          73
Controversy Level          73
Controversy Score         100
ESG Risk Percentile        73
ESG Risk Level             73
dtype: int64

Total rows: 503
Rows with ESG scores: 430
Rows missing ESG scores: 73


In [4]:
energy_df = df[df['Sector'] == 'Energy']
print(f"Energy sector companies: {len(energy_df)}")
print(energy_df['Name'].tolist())

Energy sector companies: 22
['Williams Companies Inc.', 'Valero Energy Corporation', 'Targa Resources Corp.', 'Schlumberger Limited', 'Phillips 66', 'Oneok, Inc.', 'Occidental Petroleum Corporation', 'Marathon Petroleum Corporation', 'Marathon Oil Corporation', 'Kinder Morgan, Inc.', 'Hess Corporation', 'Halliburton Company', 'Exxon Mobil Corporation', 'Eqt Corp', 'Eog Resources, Inc.', 'Diamondback Energy, Inc.', 'Devon Energy Corporation', 'Coterra Energy Inc.', 'Conocophillips', 'Chevron Corporation', 'Baker Hughes Company', 'Apa Corporation']


In [5]:
energy_df = df[df['Sector'] == 'Energy'].copy()

print(f"Total energy companies: {len(energy_df)}")
print()
print(f"Energy companies WITH ESG scores: {energy_df['Total ESG Risk score'].notna().sum()}")
print(f"Energy companies WITHOUT ESG scores: {energy_df['Total ESG Risk score'].isna().sum()}")
print()
print(energy_df[['Symbol', 'Name', 'Industry', 
                  'Total ESG Risk score', 
                  'ESG Risk Level']].to_string())

Total energy companies: 22

Energy companies WITH ESG scores: 20
Energy companies WITHOUT ESG scores: 2

    Symbol                              Name                        Industry  Total ESG Risk score ESG Risk Level
14     WMB           Williams Companies Inc.             Oil & Gas Midstream                  21.8         Medium
41     VLO         Valero Energy Corporation  Oil & Gas Refining & Marketing                  32.6           High
85    TRGP             Targa Resources Corp.             Oil & Gas Midstream                  33.2           High
108    SLB              Schlumberger Limited  Oil & Gas Equipment & Services                  20.3         Medium
146    PSX                       Phillips 66  Oil & Gas Refining & Marketing                  35.4           High
162    OKE                       Oneok, Inc.             Oil & Gas Midstream                  25.1         Medium
166    OXY  Occidental Petroleum Corporation                   Oil & Gas E&P                  41.

In [6]:
# Drop companies with missing ESG scores
energy_clean = energy_df[energy_df['Total ESG Risk score'].notna()].copy()

# Drop Controversy Score column (too many nulls)
energy_clean = energy_clean.drop(columns=['Controversy Score'])

# Reset index
energy_clean = energy_clean.reset_index(drop=True)

# Check what we have
print(f"Clean dataset shape: {energy_clean.shape}")
print()
print(energy_clean[['Symbol', 'Name', 'Industry', 
                     'Total ESG Risk score',
                     'Environment Risk Score',
                     'Social Risk Score', 
                     'Governance Risk Score',
                     'ESG Risk Level']].to_string())

Clean dataset shape: (20, 14)

   Symbol                              Name                        Industry  Total ESG Risk score  Environment Risk Score  Social Risk Score  Governance Risk Score ESG Risk Level
0     WMB           Williams Companies Inc.             Oil & Gas Midstream                  21.8                     8.7               10.1                    3.0         Medium
1     VLO         Valero Energy Corporation  Oil & Gas Refining & Marketing                  32.6                    20.1                7.4                    5.0           High
2    TRGP             Targa Resources Corp.             Oil & Gas Midstream                  33.2                    14.2               14.8                    4.2           High
3     SLB              Schlumberger Limited  Oil & Gas Equipment & Services                  20.3                     8.1                6.3                    5.9         Medium
4     PSX                       Phillips 66  Oil & Gas Refining & Marketin

In [7]:
# ============================================================
# Data Cleaning and Preparation
# ESG Risk Analyser — US Energy Sector
# Author: Sai Sajan Yerra
# ============================================================

# Check data types
print("=== Data Types ===")
print(energy_clean.dtypes)
print()

# Clean Full Time Employees column (has commas)
energy_clean['Full Time Employees'] = (
    energy_clean['Full Time Employees']
    .str.replace(',', '')
    .str.strip()
)

# Convert to numeric where possible
energy_clean['Full Time Employees'] = pd.to_numeric(
    energy_clean['Full Time Employees'], 
    errors='coerce'
)

# Check ESG Risk Level distribution
print("=== ESG Risk Level Distribution ===")
print(energy_clean['ESG Risk Level'].value_counts())
print()

# Check Industry distribution
print("=== Industry Distribution ===")
print(energy_clean['Industry'].value_counts())
print()

# Check score ranges
print("=== ESG Score Statistics ===")
print(energy_clean[[
    'Total ESG Risk score',
    'Environment Risk Score',
    'Social Risk Score',
    'Governance Risk Score'
]].describe().round(2))

=== Data Types ===
Symbol                     object
Name                       object
Address                    object
Sector                     object
Industry                   object
Full Time Employees        object
Description                object
Total ESG Risk score      float64
Environment Risk Score    float64
Governance Risk Score     float64
Social Risk Score         float64
Controversy Level          object
ESG Risk Percentile        object
ESG Risk Level             object
dtype: object

=== ESG Risk Level Distribution ===
High      13
Medium     4
Severe     2
Low        1
Name: ESG Risk Level, dtype: int64

=== Industry Distribution ===
Oil & Gas E&P                     9
Oil & Gas Midstream               4
Oil & Gas Refining & Marketing    3
Oil & Gas Equipment & Services    2
Oil & Gas Integrated              2
Name: Industry, dtype: int64

=== ESG Score Statistics ===
       Total ESG Risk score  Environment Risk Score  Social Risk Score  \
count                 2

In [8]:
# ============================================================
# Analysis Layer
# ESG Risk Analyser — US Energy Sector
# Author: Sai Sajan Yerra
# ============================================================

# Add ESG Risk Score numeric rank
energy_clean['Risk_Rank'] = energy_clean['Total ESG Risk score'].rank(
    ascending=False
).astype(int)

# Add dominant risk driver column
energy_clean['Dominant_Risk_Driver'] = energy_clean[[
    'Environment Risk Score',
    'Social Risk Score', 
    'Governance Risk Score'
]].idxmax(axis=1)

# Clean up the column name for readability
energy_clean['Dominant_Risk_Driver'] = (
    energy_clean['Dominant_Risk_Driver']
    .str.replace(' Risk Score', '')
    .str.replace(' Risk', '')
)

# Add environment risk as percentage of total
energy_clean['Environment_Pct_of_Total'] = (
    (energy_clean['Environment Risk Score'] / 
     energy_clean['Total ESG Risk score']) * 100
).round(1)

print("=== Full Analysis Dataset ===")
print(energy_clean[[
    'Symbol', 'Name', 'Industry',
    'Total ESG Risk score', 'ESG Risk Level',
    'Risk_Rank', 'Dominant_Risk_Driver',
    'Environment_Pct_of_Total'
]].sort_values('Risk_Rank').to_string())

=== Full Analysis Dataset ===
   Symbol                              Name                        Industry  Total ESG Risk score ESG Risk Level  Risk_Rank Dominant_Risk_Driver  Environment_Pct_of_Total
6     OXY  Occidental Petroleum Corporation                   Oil & Gas E&P                  41.7         Severe          1          Environment                      60.0
12    XOM           Exxon Mobil Corporation            Oil & Gas Integrated                  41.6         Severe          2          Environment                      55.5
19    APA                   Apa Corporation                   Oil & Gas E&P                  38.8           High          3          Environment                      56.7
8     MRO          Marathon Oil Corporation                   Oil & Gas E&P                  37.7           High          4          Environment                      56.0
18    CVX               Chevron Corporation            Oil & Gas Integrated                  36.6           High   

In [9]:
# Save clean analysis dataset to processed folder
energy_clean.to_csv(
    r"D:/Git/esg-disclosure-gap-analyser/data/processed/energy_esg_clean.csv", 
    index=False
)

print("File saved successfully.")
print(f"Shape: {energy_clean.shape}")
print(f"Columns: {energy_clean.columns.tolist()}")

File saved successfully.
Shape: (20, 17)
Columns: ['Symbol', 'Name', 'Address', 'Sector', 'Industry', 'Full Time Employees', 'Description', 'Total ESG Risk score', 'Environment Risk Score', 'Governance Risk Score', 'Social Risk Score', 'Controversy Level', 'ESG Risk Percentile', 'ESG Risk Level', 'Risk_Rank', 'Dominant_Risk_Driver', 'Environment_Pct_of_Total']


In [10]:
print("=== Average ESG Scores by Industry ===")
industry_summary = energy_clean.groupby('Industry')[[
    'Total ESG Risk score',
    'Environment Risk Score',
    'Social Risk Score',
    'Governance Risk Score'
]].mean().round(2).sort_values('Total ESG Risk score', ascending=False)

print(industry_summary)

=== Average ESG Scores by Industry ===
                                Total ESG Risk score  Environment Risk Score  \
Industry                                                                       
Oil & Gas Integrated                           39.10                   20.05   
Oil & Gas E&P                                  36.11                   20.06   
Oil & Gas Refining & Marketing                 32.83                   19.33   
Oil & Gas Midstream                            24.78                   10.50   
Oil & Gas Equipment & Services                 22.95                    8.95   

                                Social Risk Score  Governance Risk Score  
Industry                                                                  
Oil & Gas Integrated                         9.60                   9.45  
Oil & Gas E&P                                8.52                   7.57  
Oil & Gas Refining & Marketing               8.10                   5.37  
Oil & Gas Midstream      

In [11]:
print("=== Risk Level Distribution ===")
risk_dist = energy_clean['ESG Risk Level'].value_counts()
print(risk_dist)
print()
print(f"Percentage Severe or High: {((risk_dist.get('Severe',0) + risk_dist.get('High',0)) / len(energy_clean) * 100).round(1)}%")

=== Risk Level Distribution ===
High      13
Medium     4
Severe     2
Low        1
Name: ESG Risk Level, dtype: int64

Percentage Severe or High: 75.0%


In [12]:
print("=== TOP 5 HIGHEST ESG RISK ===")
print(energy_clean.nlargest(5, 'Total ESG Risk score')[
    ['Name', 'Industry', 'Total ESG Risk score', 'ESG Risk Level']
].to_string())

print()
print("=== TOP 5 LOWEST ESG RISK ===")
print(energy_clean.nsmallest(5, 'Total ESG Risk score')[
    ['Name', 'Industry', 'Total ESG Risk score', 'ESG Risk Level']
].to_string())

=== TOP 5 HIGHEST ESG RISK ===
                                Name              Industry  Total ESG Risk score ESG Risk Level
6   Occidental Petroleum Corporation         Oil & Gas E&P                  41.7         Severe
12           Exxon Mobil Corporation  Oil & Gas Integrated                  41.6         Severe
19                   Apa Corporation         Oil & Gas E&P                  38.8           High
8           Marathon Oil Corporation         Oil & Gas E&P                  37.7           High
18               Chevron Corporation  Oil & Gas Integrated                  36.6           High

=== TOP 5 LOWEST ESG RISK ===
                       Name                        Industry  Total ESG Risk score ESG Risk Level
9       Kinder Morgan, Inc.             Oil & Gas Midstream                  19.0            Low
3      Schlumberger Limited  Oil & Gas Equipment & Services                  20.3         Medium
0   Williams Companies Inc.             Oil & Gas Midstream            

In [13]:
print("=== Environment Risk as % of Total by Industry ===")
env_pct = energy_clean.groupby('Industry')[
    'Environment_Pct_of_Total'
].mean().round(1).sort_values(ascending=False)
print(env_pct)

=== Environment Risk as % of Total by Industry ===
Industry
Oil & Gas Refining & Marketing    58.9
Oil & Gas E&P                     55.4
Oil & Gas Integrated              51.0
Oil & Gas Midstream               42.4
Oil & Gas Equipment & Services    39.1
Name: Environment_Pct_of_Total, dtype: float64


In [14]:
pip install psycopg2-binary


Note: you may need to restart the kernel to use updated packages.


In [16]:
import psycopg2
import pandas as pd

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    port="5432",
    database="esg_energy",
    user="postgres",
    password="admin"
)

cursor = conn.cursor()
print("Connected to PostgreSQL successfully.")
print(f"PostgreSQL version: {conn.server_version}")

Connected to PostgreSQL successfully.
PostgreSQL version: 180003


In [17]:
# ============================================================
# Create Tables in PostgreSQL
# ESG Risk Analyser — US Energy Sector
# Author: Sai Sajan Yerra
# ============================================================

cursor.execute("""
    DROP TABLE IF EXISTS esg_scores;
    DROP TABLE IF EXISTS industry_summary;
    DROP TABLE IF EXISTS companies;
""")

cursor.execute("""
    CREATE TABLE companies (
        id                  SERIAL PRIMARY KEY,
        symbol              TEXT NOT NULL,
        name                TEXT NOT NULL,
        sector              TEXT,
        industry            TEXT,
        full_time_employees INTEGER,
        address             TEXT,
        description         TEXT
    );
""")

cursor.execute("""
    CREATE TABLE esg_scores (
        id                          SERIAL PRIMARY KEY,
        company_id                  INTEGER NOT NULL,
        total_esg_risk_score        REAL,
        environment_risk_score      REAL,
        social_risk_score           REAL,
        governance_risk_score       REAL,
        controversy_level           TEXT,
        esg_risk_percentile         TEXT,
        esg_risk_level              TEXT,
        risk_rank                   INTEGER,
        dominant_risk_driver        TEXT,
        environment_pct_of_total    REAL,
        FOREIGN KEY (company_id) REFERENCES companies(id)
    );
""")

cursor.execute("""
    CREATE TABLE industry_summary (
        id                          SERIAL PRIMARY KEY,
        industry                    TEXT NOT NULL,
        avg_total_esg_risk_score    REAL,
        avg_environment_risk_score  REAL,
        avg_social_risk_score       REAL,
        avg_governance_risk_score   REAL,
        company_count               INTEGER
    );
""")

conn.commit()
print("All tables created successfully.")

All tables created successfully.


In [18]:
for _, row in energy_clean.iterrows():
    cursor.execute("""
        INSERT INTO companies 
        (symbol, name, sector, industry, 
         full_time_employees, address, description)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (
        row['Symbol'],
        row['Name'],
        row['Sector'],
        row['Industry'],
        int(row['Full Time Employees']) if pd.notna(row['Full Time Employees']) else None,
        row['Address'] if pd.notna(row['Address']) else None,
        row['Description'] if pd.notna(row['Description']) else None
    ))

conn.commit()
print(f"Inserted {len(energy_clean)} companies.")

# Verify
result = pd.read_sql("SELECT id, symbol, name, industry FROM companies", conn)
print(result.to_string())

Inserted 20 companies.
    id symbol                              name                        industry
0    1    WMB           Williams Companies Inc.             Oil & Gas Midstream
1    2    VLO         Valero Energy Corporation  Oil & Gas Refining & Marketing
2    3   TRGP             Targa Resources Corp.             Oil & Gas Midstream
3    4    SLB              Schlumberger Limited  Oil & Gas Equipment & Services
4    5    PSX                       Phillips 66  Oil & Gas Refining & Marketing
5    6    OKE                       Oneok, Inc.             Oil & Gas Midstream
6    7    OXY  Occidental Petroleum Corporation                   Oil & Gas E&P
7    8    MPC    Marathon Petroleum Corporation  Oil & Gas Refining & Marketing
8    9    MRO          Marathon Oil Corporation                   Oil & Gas E&P
9   10    KMI               Kinder Morgan, Inc.             Oil & Gas Midstream
10  11    HES                  Hess Corporation                   Oil & Gas E&P
11  12    HAL    

C:\Users\sajan\AppData\Local\Temp\ipykernel_44560\1458551887.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql("SELECT id, symbol, name, industry FROM companies", conn)


In [19]:
# Get company id mapping
company_ids = pd.read_sql("SELECT id, symbol FROM companies", conn)
company_id_map = dict(zip(company_ids['symbol'], company_ids['id']))

for _, row in energy_clean.iterrows():
    cid = company_id_map[row['Symbol']]
    cursor.execute("""
        INSERT INTO esg_scores
        (company_id, total_esg_risk_score, environment_risk_score,
         social_risk_score, governance_risk_score, controversy_level,
         esg_risk_percentile, esg_risk_level, risk_rank,
         dominant_risk_driver, environment_pct_of_total)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        cid,
        row['Total ESG Risk score'],
        row['Environment Risk Score'],
        row['Social Risk Score'],
        row['Governance Risk Score'],
        row['Controversy Level'] if pd.notna(row['Controversy Level']) else None,
        row['ESG Risk Percentile'] if pd.notna(row['ESG Risk Percentile']) else None,
        row['ESG Risk Level'],
        int(row['Risk_Rank']),
        row['Dominant_Risk_Driver'],
        row['Environment_Pct_of_Total']
    ))

conn.commit()
print(f"Inserted ESG scores for {len(energy_clean)} companies.")

Inserted ESG scores for 20 companies.


C:\Users\sajan\AppData\Local\Temp\ipykernel_44560\4069664695.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  company_ids = pd.read_sql("SELECT id, symbol FROM companies", conn)


In [20]:
industry_summary = energy_clean.groupby('Industry').agg(
    avg_total_esg_risk_score=('Total ESG Risk score', 'mean'),
    avg_environment_risk_score=('Environment Risk Score', 'mean'),
    avg_social_risk_score=('Social Risk Score', 'mean'),
    avg_governance_risk_score=('Governance Risk Score', 'mean'),
    company_count=('Symbol', 'count')
).round(2).reset_index()

for _, row in industry_summary.iterrows():
    cursor.execute("""
        INSERT INTO industry_summary
        (industry, avg_total_esg_risk_score, avg_environment_risk_score,
         avg_social_risk_score, avg_governance_risk_score, company_count)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        row['Industry'],
        row['avg_total_esg_risk_score'],
        row['avg_environment_risk_score'],
        row['avg_social_risk_score'],
        row['avg_governance_risk_score'],
        int(row['company_count'])
    ))

conn.commit()
print("Industry summary inserted.")
print(industry_summary.to_string())

Industry summary inserted.
                         Industry  avg_total_esg_risk_score  avg_environment_risk_score  avg_social_risk_score  avg_governance_risk_score  company_count
0                   Oil & Gas E&P                     36.11                       20.06                   8.52                       7.57              9
1  Oil & Gas Equipment & Services                     22.95                        8.95                   7.60                       6.40              2
2            Oil & Gas Integrated                     39.10                       20.05                   9.60                       9.45              2
3             Oil & Gas Midstream                     24.78                       10.50                  10.78                       3.50              4
4  Oil & Gas Refining & Marketing                     32.83                       19.33                   8.10                       5.37              3


In [21]:
print("=== QUERY 1: All companies ranked by ESG risk ===")
q1 = pd.read_sql("""
    SELECT 
        c.symbol,
        c.name,
        c.industry,
        s.total_esg_risk_score,
        s.esg_risk_level,
        s.risk_rank,
        s.dominant_risk_driver
    FROM esg_scores s
    JOIN companies c ON c.id = s.company_id
    ORDER BY s.risk_rank ASC
""", conn)
print(q1.to_string())

print()
print("=== QUERY 2: Average scores by industry ===")
q2 = pd.read_sql("""
    SELECT 
        industry,
        avg_total_esg_risk_score,
        avg_environment_risk_score,
        avg_social_risk_score,
        avg_governance_risk_score,
        company_count
    FROM industry_summary
    ORDER BY avg_total_esg_risk_score DESC
""", conn)
print(q2.to_string())

print()
print("=== QUERY 3: Companies with Severe or High risk ===")
q3 = pd.read_sql("""
    SELECT 
        c.name,
        c.industry,
        s.total_esg_risk_score,
        s.esg_risk_level,
        s.dominant_risk_driver
    FROM esg_scores s
    JOIN companies c ON c.id = s.company_id
    WHERE s.esg_risk_level IN ('Severe', 'High')
    ORDER BY s.total_esg_risk_score DESC
""", conn)
print(q3.to_string())

print()
print("=== QUERY 4: Dominant risk driver breakdown ===")
q4 = pd.read_sql("""
    SELECT 
        dominant_risk_driver,
        COUNT(*) as company_count,
        ROUND(AVG(total_esg_risk_score)::numeric, 2) as avg_score
    FROM esg_scores
    GROUP BY dominant_risk_driver
    ORDER BY company_count DESC
""", conn)
print(q4.to_string())

print()
print("=== QUERY 5: Top 5 highest ESG risk companies ===")
q5 = pd.read_sql("""
    SELECT 
        c.name,
        c.industry,
        s.total_esg_risk_score,
        s.environment_risk_score,
        s.social_risk_score,
        s.governance_risk_score,
        s.esg_risk_level
    FROM esg_scores s
    JOIN companies c ON c.id = s.company_id
    ORDER BY s.total_esg_risk_score DESC
    LIMIT 5
""", conn)
print(q5.to_string())

print()
print("=== QUERY 6: Top 5 lowest ESG risk companies ===")
q6 = pd.read_sql("""
    SELECT 
        c.name,
        c.industry,
        s.total_esg_risk_score,
        s.environment_risk_score,
        s.social_risk_score,
        s.governance_risk_score,
        s.esg_risk_level
    FROM esg_scores s
    JOIN companies c ON c.id = s.company_id
    ORDER BY s.total_esg_risk_score ASC
    LIMIT 5
""", conn)
print(q6.to_string())

# Close connection
conn.close()
print()
print("Connection closed.")
print("PostgreSQL database fully loaded and verified.")

=== QUERY 1: All companies ranked by ESG risk ===
   symbol                              name                        industry  total_esg_risk_score esg_risk_level  risk_rank dominant_risk_driver
0     OXY  Occidental Petroleum Corporation                   Oil & Gas E&P                  41.7         Severe          1          Environment
1     XOM           Exxon Mobil Corporation            Oil & Gas Integrated                  41.6         Severe          2          Environment
2     APA                   Apa Corporation                   Oil & Gas E&P                  38.8           High          3          Environment
3     MRO          Marathon Oil Corporation                   Oil & Gas E&P                  37.7           High          4          Environment
4     CVX               Chevron Corporation            Oil & Gas Integrated                  36.6           High          5          Environment
5    CTRA               Coterra Energy Inc.                   Oil & Gas E&P     

C:\Users\sajan\AppData\Local\Temp\ipykernel_44560\4237337648.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q1 = pd.read_sql("""
C:\Users\sajan\AppData\Local\Temp\ipykernel_44560\4237337648.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q2 = pd.read_sql("""
C:\Users\sajan\AppData\Local\Temp\ipykernel_44560\4237337648.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q3 = pd.read_sql("""
C:\Users\sajan\AppData\Local\Temp\ipykernel_44560\4237337648.py:50: UserWarning: pandas only supports SQLAlchemy connectable 

In [23]:
# Reconnect to database
conn = psycopg2.connect(
    host="localhost",
    port="5432",
    database="esg_energy",
    user="postgres",
    password="admin"
)

# Export full joined dataset for Tableau
tableau_df = pd.read_sql("""
    SELECT 
        c.symbol,
        c.name,
        c.sector,
        c.industry,
        c.full_time_employees,
        s.total_esg_risk_score,
        s.environment_risk_score,
        s.social_risk_score,
        s.governance_risk_score,
        s.controversy_level,
        s.esg_risk_percentile,
        s.esg_risk_level,
        s.risk_rank,
        s.dominant_risk_driver,
        s.environment_pct_of_total
    FROM esg_scores s
    JOIN companies c ON c.id = s.company_id
    ORDER BY s.risk_rank ASC
""", conn)

# Save to processed folder
tableau_df.to_csv(
    r"D:/Git/esg-disclosure-gap-analyser/data/processed/tableau_esg_energy.csv",
    index=False
)

conn.close()

print(f"Tableau export saved.")
print(f"Shape: {tableau_df.shape}")
print(tableau_df.head())

Tableau export saved.
Shape: (20, 15)
  symbol                              name  sector              industry  \
0    OXY  Occidental Petroleum Corporation  Energy         Oil & Gas E&P   
1    XOM           Exxon Mobil Corporation  Energy  Oil & Gas Integrated   
2    APA                   Apa Corporation  Energy         Oil & Gas E&P   
3    MRO          Marathon Oil Corporation  Energy         Oil & Gas E&P   
4    CVX               Chevron Corporation  Energy  Oil & Gas Integrated   

   full_time_employees  total_esg_risk_score  environment_risk_score  \
0                12570                  41.7                    25.0   
1                62000                  41.6                    23.1   
2                 2271                  38.8                    22.0   
3                 1681                  37.7                    21.1   
4                45600                  36.6                    17.0   

   social_risk_score  governance_risk_score              controversy_lev

C:\Users\sajan\AppData\Local\Temp\ipykernel_44560\1696164328.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tableau_df = pd.read_sql("""
